# Two-Tower Model — complementary products

**The goal of this notebook is to build a two-tower neural network (TTN) that
finds complementary products** — given an item a user is looking at, retrieve
the items that are bought *alongside* it rather than the items most similar to
it. A phone case complements a phone; another phone does not.

The approach follows **[Suggest, complement, inspire: story of Two Tower
recommendations at Allegro.com](https://arxiv.org/html/2508.03702v1)**
(Osowska-Kurczab, Nazarko, Marzec, Wojciechowska & Kremeňová, RecSys '25),
whose Complementary-TT model is the architecture this work is based on.

## Both towers describe items

This is the part that differs from the classic user/item two-tower setup, and
it shapes every column decision below: **the query tower and the candidate
tower both consume item information.** Neither tower is a user tower.

```
   query ITEM features                    candidate ITEM features
        │                                          │
   ┌────▼────┐                                ┌────▼────┐
   │  QUERY  │  product encoder               │CANDIDATE│  product encoder
   │  TOWER  │  (+ target category)           │  TOWER  │
   └────┬────┘                                └────┬────┘
        │                                          │
   q ∈ ℝ^d  ──────────  score = q · c  ──────────  c ∈ ℝ^d
```

Both towers share the same *architecture* — the paper's "Product Encoder":
each item attribute goes through its own embedding table, the vectors are
concatenated, passed through an MLP and L2-normalised. In the paper the query
tower is the only one modified for the complementary task: the query product
embedding is concatenated with a **target category embedding** drawn from a
one-to-many complementary-category mapping, while the candidate tower stays a
plain product encoder.

That mapping is what `complementary_cats_pairs/` produces —
`data/complementary_categories.pkl`, source category path → target category path,
scored by support and lift. §7 loads it. The co-purchase pairs that supply the
training positives come from the same package's `pairs.ipynb`.

Because both sides are items, the user features assembled in §4 are **not**
tower inputs here. They are kept because this table is also the base for the
user/item variant, and because a user's history is what defines which pairs
count as co-purchased in the first place.

## What this notebook covers

It assembles and time-splits the dataset the model trains on. Three sources,
one row per interaction:

| Source | Grain | Joined how |
| --- | --- | --- |
| `Home_and_Kitchen_filtered.csv` | one row per review | the base table |
| `df_features.pkl` | one row per `asin` | left join on `asin` |
| `df_user_features.pkl` | one row per purchase event | positional concat |

Model definition and training come next, once the columns are settled — §6 is
where the split that training depends on is set.


In [16]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# Show every column/variable when displaying a dataframe
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 50)
# Turn off scientific notation (e.g. 2.447268e+06 -> 2447268.00)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Load the user–item interactions

`data/Home_and_Kitchen_filtered.csv` is the interaction log: **one row per
review**, i.e. one row per (user, item, date) event. This is the table the
two-tower model is trained on — every other dataset in this notebook is joined
onto it.

**All 11 columns are loaded here** so the full contents are visible before
anything is thrown away. The next section decides what to keep.

`asin` and `reviewerID` are pinned to `str` so IDs with leading zeros
(e.g. `0560467893`) survive, and `low_memory=False` avoids the mixed-type
warning on `vote`.

In [17]:
from pathlib import Path

# data/ lives at the repo root, one level up from this ttn/ folder
DATA_DIR = Path("..") / "data"

df_reviews = pd.read_csv(
    DATA_DIR / "Home_and_Kitchen_filtered.csv",
    dtype={"asin": str, "reviewerID": str},
    low_memory=False,
)

print("df_reviews:", df_reviews.shape)
print("columns:", list(df_reviews.columns))
print(f"unique users: {df_reviews['reviewerID'].nunique():,} | "
      f"unique items: {df_reviews['asin'].nunique():,}")
df_reviews.head(5)

df_reviews: (6898955, 11)
columns: ['overall', 'verified', 'reviewTime', 'reviewerID', 'asin', 'reviewerName', 'summary', 'unixReviewTime', 'vote', 'style', 'image']
unique users: 777,242 | unique items: 189,172


,overall,verified,reviewTime,reviewerID,asin,reviewerName,summary,unixReviewTime,vote,style,image
0,5.00,True,"11 5, 2015",A8LUWTIPU9CZB,0560467893,Linda Fahner,Five Stars,1446681600,NaN,NaN,NaN
1,3.00,True,"05 7, 2015",A3B6GKQQ1JJ167,0560467893,Harry Slaughter,Meh,1430956800,2,NaN,NaN
2,5.00,True,"01 22, 2014",A3MCTN65BU7XRA,0681795107,luckyg,Recommend,1390348800,NaN,{'Color:': ' Brushed Stainless'},NaN
3,1.00,True,"10 30, 2013",A7JVZFSXVY9RL,0681795107,Nickleen,Not keeping coffee hot for long enough,1383091200,NaN,{'Color:': ' Brushed Stainless'},NaN
4,1.00,True,"09 20, 2013",A2RQ7VLAK1SHPU,0681795107,Lacemaker427,Leaks like a waterfall when at an angle!,1379635200,NaN,{'Color:': ' Red'},NaN


### Drop what the model cannot use at serving time

Two reasons to drop a column, and it matters which applies:

**(a) Post-interaction.** `overall`, `vote`, `summary`, `image` only exist
*after* the purchase. At recommendation time we don't have them, so training on
them learns from information that will never be there in production. Training
here is implicit — the interaction itself is the positive signal — so the rating
is neither needed as a label nor usable as an input.

**(b) No signal.** `reviewerName` is a display name, not an identifier;
`reviewerID` already identifies the user. `style` describes a *variant* while
the model recommends at `asin` level, and it is a high-cardinality dict string.

Kept: `reviewerID` + `asin` (the interaction), `unixReviewTime` (the timeline
and the split key), `reviewTime` (readable duplicate), `verified` (a property of
the transaction, so known at interaction time).

⚠️ `style` and `summary` are what separate a genuine same-day purchase of two
variants from the same review recorded twice. If you de-duplicate the log, do it
**before** this cell — 231,139 rows are exact duplicates across all 11 columns,
but deduping after this drop collapses ~17k real interactions too.

In [18]:
# (a) post-interaction — known only after the purchase, would leak at serving time
POST_INTERACTION = ["overall", "vote", "summary", "image"]

# (b) no signal / redundant
NO_SIGNAL = ["reviewerName", "style"]

DROP_COLS = POST_INTERACTION + NO_SIGNAL

df_reviews = df_reviews.drop(columns=[c for c in DROP_COLS if c in df_reviews.columns])

print(f"dropped {len(DROP_COLS)}: {DROP_COLS}")
print(f"kept    {df_reviews.shape[1]}: {list(df_reviews.columns)}")
print("\ndf_reviews:", df_reviews.shape)
df_reviews.head(5)

dropped 6: ['overall', 'vote', 'summary', 'image', 'reviewerName', 'style']
kept    5: ['verified', 'reviewTime', 'reviewerID', 'asin', 'unixReviewTime']

df_reviews: (6898955, 5)


,verified,reviewTime,reviewerID,asin,unixReviewTime
0,True,"11 5, 2015",A8LUWTIPU9CZB,0560467893,1446681600
1,True,"05 7, 2015",A3B6GKQQ1JJ167,0560467893,1430956800
2,True,"01 22, 2014",A3MCTN65BU7XRA,0681795107,1390348800
3,True,"10 30, 2013",A7JVZFSXVY9RL,0681795107,1383091200
4,True,"09 20, 2013",A2RQ7VLAK1SHPU,0681795107,1379635200


## 2. Item features

`data/df_features.pkl` — one row per `asin` (~1.13M items) with the attributes
extracted by `feature_extraction_workflow/extract_features.py`: `cat_*`,
`brand`, the per-field columns (`Product_Type`, `Material`, `Color`, …), their
parsed `_numeric` / `_unit` / `_cleaned` measures, and `title_cleaned`.

It is loaded and **analysed on its own first**. The join onto the interactions
happens in §4, once the checks below have settled which columns are worth
carrying.

In [19]:
from pathlib import Path

# data/ lives at the repo root, one level up from this ttn/ folder
DATA_DIR = Path("..") / "data"

# --- Item features (one row per asin) ---
df_features = pd.read_pickle(DATA_DIR / "df_features.pkl")
print("df_features:", df_features.shape)
df_features.head(3)

df_features: (1134566, 92)


,category,tech1,description,title,tech2,brand,also_buy,feature,rank,main_cat,price,asin,date,imageURL,imageURLHighRes,cat_1,cat_2,cat_3,cat_4,cat_5,cat_6,title_cleaned,extracted_features_title,description_cleaned,extracted_features_description,feature_cleaned,extracted_features_feature,extracted_features,Bar_Pressure,Brand,Capacity_Cups,Capacity_Volume,Color,Density_Weight,Dimensions,Features,Filter_Rating,Material,Part_Number,Piece_Count,Pocket_Depth,Power_Rating,Product_Type,Scent,Shape,Shape_Style,Size,Stage_Count,Sub_Type,Theme,Thread_Count,Voltage,Weight,capacity_volume_numeric,capacity_volume_unit,piece_count_numeric,piece_count_unit,thread_count_numeric,thread_count_unit,weight_numeric,weight_unit,bar_pressure_numeric,capacity_cups_numeric,density_weight_lb,pocket_depth_in,power_rating_w,stage_count_numeric,voltage_numeric,dimension_1,dimension_2,dimension_3,dimension_unit,brand_clean,dimension_1_in,dimension_2_in,dimension_3_in,dimension_unit_src,dimension_unit_clean,bar_pressure_numeric_cleaned,capacity_cups_numeric_cleaned,density_weight_lb_cleaned,pocket_depth_in_cleaned,power_rating_w_cleaned,stage_count_numeric_cleaned,voltage_numeric_cleaned,capacity_volume_numeric_cleaned,weight_numeric_cleaned,piece_count_numeric_cleaned,thread_count_numeric_cleaned,dimension_1_in_cleaned,dimension_2_in_cleaned,dimension_3_in_cleaned
0,"['Home & Kitchen', 'Kitchen & Dining', 'Dining...",NaN,['It was a time honored tradition among the ea...,You Are Special Today Red Plate [With Red Pen],NaN,Waechtersbach USA,"['B0001XR2F2', 'B01LY51HUN', 'B07CXZ9C5B', '03...",[],"['>#39,665 in Kitchen & Dining (See Top 100 in...",Amazon Home,$37.00,0001487795,"October 8, 2006",[],[],Home & Kitchen,Kitchen & Dining,Dining & Entertaining,Dinnerware,Plates,Dinner Plates,special today red plate red pen,{'Color': 'red'},time honored tradition among early american fa...,{'Color': 'red'},,{},{'Color': 'red'},None,None,None,None,red,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,other_brands,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"['Home & Kitchen', 'Home Dcor', 'Candles & Hol...",NaN,['VICKS INHALER relieves stuffy noses helps si...,Vicks Inhaler Relief for Cold Sinus Nasal Cong...,NaN,Vicks,[],[],"['>#1,763,185 in Home & Kitchen (See Top 100 i...",Amazon Home,$4.05,0002020300,NaN,[],[],Home & Kitchen,Home Dcor,Candles & Holders,Candles,None,None,vicks inhaler relief cold sinus nasal congesti...,{},vicks inhaler relief stuffy nose help sinus co...,{},,{},{},None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,other_brands,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"['Home & Kitchen', 'Kitchen & Dining', 'Dining...",NaN,"['16 oz squeeze bottle, 1 lb.']",Artistic Churchware Communion Cup Filler: RW525,NaN,Artistic Churchware,[],"['Religious Supply Center', 'RW-525', 'Communi...","['>#2,127,003 in Home & Kitchen (See Top 100 i...",Amazon Home,$12.48,0006564224,NaN,[],[],Home & Kitchen,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,Wine & Champagne Glasses,None,artistic churchware communion cup filler rw525,{'Product_Type': 'cup'},16 oz squeeze bottle 1 lb,{'Capacity_Volume': '16 oz'},religious supply center rw-525 communion cup f...,{'Product_Type': 'cup'},"{'Product_Type': 'cup', 'Capacity_Volume': '16...",None,None,None,16 oz,None,None,None,None,None,None,None,None,None,None,cup,None,None,None,None,None,None,None,None,None,None,16.00,oz,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,other_brands,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.00,NaN,NaN,NaN,NaN,NaN,NaN


### Validate the feature table

Before anything is joined, check that `df_features.pkl` is what the pipeline
promised: the exact expected column set, one row per `asin`, numeric columns
actually numeric, coverage above its floors, unit columns free of new values,
and every `_cleaned` column inside its bound.

A **FAIL on `columns`** is the one to care about most — it means a feature
appeared that nothing describes, or one silently disappeared.

In [20]:
import sys
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from feature_extraction_workflow.validations import run_all

report = run_all(df_features, DATA_DIR / "master_metadata.json")
display(report if len(report) else "no findings")

1,134,566 rows x 92 columns — 1 failure(s), 0 warning(s)


,check,level,subject,detail
0,columns,FAIL,also_buy,present in the table but not in the contract


### Two cleaning steps that still live here

`cat_4_clean` is built from `data/category_taxonomy.json` — a reviewed
whitelist of which `(cat_3, cat_4)` pairs are real categories rather than
product bullets that leaked into the path. 921 → 451 distinct values, with
0.1% of items landing in a `<cat_3>_Other` bucket. §3 joins `cat_4_clean`
rather than raw `cat_4`.

`brand_clean` folds brands carried by ten or fewer items into `other_brands`.

Both are also produced by the pipeline now (`clean_brand` is Filter 5), so once
you re-extract, the brand cell here is recomputing what the pickle already
carries.

In [21]:
import json

TAXONOMY_PATH = DATA_DIR / "category_taxonomy.json"
with open(TAXONOMY_PATH) as f:
    taxonomy = json.load(f)

# cat_3 -> the set of cat_4 values that survived the review
valid_cat_4 = {c3: set(vals) for c2 in taxonomy for c3, vals in taxonomy[c2].items()}
print(f"taxonomy: {len(taxonomy)} cat_2 | {len(valid_cat_4)} cat_3 | "
      f"{sum(len(v) for v in valid_cat_4.values())} valid cat_4 slots")

MISSING = "Missing"
OTHER_SUFFIX = "_Other"

cat_3 = df_features["cat_3"].astype(str)
cat_4 = df_features["cat_4"].fillna(MISSING).astype(str)

# A value is kept only if it is valid *under its own parent* — the same label
# can be real in one branch and junk in another.
valid_pairs = {(c3, v) for c3, vals in valid_cat_4.items() for v in vals}
keep = pd.Series(list(zip(cat_3, cat_4)), index=df_features.index).isin(valid_pairs)

df_features["cat_4_clean"] = np.where(keep, cat_4, cat_3 + OTHER_SUFFIX)

n_before = df_features["cat_4"].nunique(dropna=False)
n_after = df_features["cat_4_clean"].nunique()
n_folded = int((~keep).sum())
print(f"\ndistinct cat_4 : {n_before:,} -> {n_after:,}")
print(f"items folded into '<cat_3>{OTHER_SUFFIX}': {n_folded:,} "
      f"({n_folded / len(df_features) * 100:.2f}% of the catalog)")
df_features[["asin", "cat_2", "cat_3", "cat_4", "cat_4_clean"]].head(5)

taxonomy: 7 cat_2 | 69 cat_3 | 521 valid cat_4 slots

distinct cat_4 : 921 -> 451
items folded into '<cat_3>_Other': 1,099 (0.10% of the catalog)


,asin,cat_2,cat_3,cat_4,cat_4_clean
0,0001487795,Kitchen & Dining,Dining & Entertaining,Dinnerware,Dinnerware
1,0002020300,Home Dcor,Candles & Holders,Candles,Candles
2,0006564224,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,Glassware & Drinkware
3,0009046461,Bath,Bathroom Accessories,None,Missing
4,0234937912,Home Dcor,Home Fragrance,Incense & Incense Holders,Incense & Incense Holders


In [22]:
# Fold rare brands: keep those carried by MORE THAN 10 distinct items.
# Counted on brand_norm (from 3.4), not the raw column — otherwise "3d rose" and
# "3drose" are counted separately and a brand can fall under the threshold only
# because its spelling is split.
MIN_ITEMS = 10
OTHER = "other_brands"

source = "brand_norm" if "brand_norm" in df_features.columns else "brand"
brand_counts = df_features.groupby(source)["asin"].nunique()

kept = brand_counts[brand_counts > MIN_ITEMS].index
df_features["brand_clean"] = df_features[source].where(
    df_features[source].isin(kept) | df_features[source].isna(),
    OTHER,
)

n_before = df_features[source].nunique()
n_after = df_features["brand_clean"].nunique()
n_missing = df_features[source].isna().sum()
print(f"counted on : {source}")
print(f"brands     : {n_before:,} -> {n_after:,} "
      f"(kept {len(kept):,} with > {MIN_ITEMS} items, rest -> {OTHER!r})")
print(f"items       : {(df_features['brand_clean'] == OTHER).mean():.1%} in {OTHER}, "
      f"{n_missing / len(df_features):.1%} missing (left as NaN)")
print()
print(df_features["brand_clean"].value_counts().head(20))

# --- Choosing the threshold ------------------------------------------------
# The cut above keeps brands carried by >= 10 items. What the model actually
# sees is INTERACTIONS, though: a brand on 3 items with 8,000 reviews has
# plenty of signal, while one on 40 items with 45 reviews gives its embedding
# ~1 gradient update per row. Judge a threshold by the share of the data that
# keeps a real brand, not by how small the table gets.

def brand_coverage(counts, label, thresholds=(1, 5, 10, 20, 50, 100, 500, 1000)):
    total_units, total_brands = counts.sum(), len(counts)
    rows = []
    for n in thresholds:
        keep = counts[counts >= n]
        rows.append({"min_" + label: n,
                     "brands_kept": len(keep),
                     "brands_kept_%": len(keep) / total_brands * 100,
                     f"{label}_covered_%": keep.sum() / total_units * 100})
    return pd.DataFrame(rows).set_index("min_" + label)

print("\nby ITEMS (what df_features can measure):")
display(brand_coverage(brand_counts, "items").round(2))

# The interaction view needs the merged frame from section 4; run this again
# after the join to pick the threshold on the number that matters.
if "df" in dir() and "brand" in getattr(df, "columns", []):
    print("by INTERACTIONS (post-join):")
    display(brand_coverage(df.groupby("brand").size(), "interactions").round(2))
else:
    print("by INTERACTIONS: run again after the join in section 4 "
          "(`df` not built yet)")

counted on : brand
brands     : 98,532 -> 12,747 (kept 12,746 with > 10 items, rest -> 'other_brands')
items       : 16.5% in other_brands, 5.7% missing (left as NaN)

brand_clean
other_brands                187662
3dRose                        8569
CafePress                     7489
Disney                        6249
Unknown                       5594
Hallmark                      4720
Generic                       4500
Department 56                 3416
Safavieh                      3227
Kurt Adler                    3124
Enesco                        2992
Wilton                        2917
Lenox                         2782
Pop Culture Graphics          2760
Coaster Home Furnishings      2604
Tervis                        2332
Tupperware                    2330
Tree-Free Greetings           2269
Trademark Fine Art            2237
Cuisinart                     2198
Name: count, dtype: int64

by ITEMS (what df_features can measure):


,brands_kept,brands_kept_%,items_covered_%
min_items,,,
1,98532,100.00,100.00
5,23357,23.70,89.25
10,13719,13.92,83.37
20,7944,8.06,76.09
50,3536,3.59,63.49
100,1750,1.78,51.90
500,228,0.23,23.61
1000,75,0.08,13.75


by INTERACTIONS (post-join):


,brands_kept,brands_kept_%,interactions_covered_%
min_interactions,,,
1,27909,100.00,100.00
5,27860,99.82,100.00
10,20600,73.81,99.18
20,14860,53.24,97.80
50,9125,32.70,94.66
100,5892,21.11,90.67
500,1760,6.31,74.90
1000,944,3.38,64.78


## 3. Join the item features onto the interactions

With the examination above settled, attach the item columns to the interaction
table on `asin`. Left join, so no interaction is dropped: items with no
extracted features keep their row with NaN item columns (~16% of rows —
those asins either aren't in the item metadata, or were dropped during
extraction because their `cat_3` has no schema).

The cell below currently attaches **every** field and measure column. Once §3
has produced a keep-list, narrow `keep_cols` to it — that is the one place the
exclusion decision needs to be applied.

In [23]:
# --- Connect the two on `asin` ---
# Attach ALL extracted feature variables plus their parsed measures.

# 1. Every extracted feature field column (the keys present in the dicts):
#    Product_Type, Material, Color, Weight, Dimensions, Brand, Theme, ...
field_cols = sorted({
    k for d in df_features["extracted_features"]
    if isinstance(d, dict) for k in d
})

# 2. Their parsed measures: numeric value + unit (+ dimension_1/2/3).
#    Use the range-cleaned numeric variant wherever one exists.
measure_cols = [
    c for c in df_features.columns
    if c.endswith("_numeric") or c.endswith("_unit")
    or c.startswith("dimension_")
    or c in ("density_weight_lb", "pocket_depth_in", "power_rating_w")
]
measure_cols = sorted({
    f"{c}_cleaned" if f"{c}_cleaned" in df_features.columns else c
    for c in measure_cols
})

# 3. Context columns to carry along (asin is the join key).
# cat_4_clean (from category_taxonomy.json) replaces the raw cat_4 here —
# the ~920 raw values are mostly bullet text, see section 3.
context_cols = ["asin", "cat_2", "cat_3", "cat_4_clean", "brand", "extracted_features"]

keep_cols = list(dict.fromkeys(context_cols + field_cols + measure_cols))
keep_cols = [c for c in keep_cols if c in df_features.columns]

df = df_reviews.merge(
    df_features[keep_cols],
    on="asin",
    how="left",
    validate="many_to_one",   # many reviews -> one item row
    indicator=True,
)
n_unmatched = (df["_merge"] == "left_only").sum()
df = df.drop(columns="_merge")

print(f"merged: {df.shape}  ({len(keep_cols)} item cols attached)")
print(f"  feature fields : {len(field_cols)}")
print(f"  measure cols   : {len(measure_cols)}")
print(f"unique users: {df['reviewerID'].nunique():,} | "
      f"unique items: {df['asin'].nunique():,}")
print(f"reviews with no matching item features: {n_unmatched:,}")
df.head(5)

merged: (6898955, 59)  (55 item cols attached)
  feature fields : 25
  measure cols   : 24
unique users: 777,242 | unique items: 189,172
reviews with no matching item features: 1,134,069


,verified,reviewTime,reviewerID,asin,unixReviewTime,cat_2,cat_3,cat_4_clean,brand,extracted_features,Bar_Pressure,Brand,Capacity_Cups,Capacity_Volume,Color,Density_Weight,Dimensions,Features,Filter_Rating,Material,Part_Number,Piece_Count,Pocket_Depth,Power_Rating,Product_Type,Scent,Shape,Shape_Style,Size,Stage_Count,Sub_Type,Theme,Thread_Count,Voltage,Weight,bar_pressure_numeric_cleaned,capacity_cups_numeric_cleaned,capacity_volume_numeric_cleaned,capacity_volume_unit,density_weight_lb_cleaned,dimension_1,dimension_1_in_cleaned,dimension_2,dimension_2_in_cleaned,dimension_3,dimension_3_in_cleaned,dimension_unit,dimension_unit_clean,dimension_unit_src,piece_count_numeric_cleaned,piece_count_unit,pocket_depth_in_cleaned,power_rating_w_cleaned,stage_count_numeric_cleaned,thread_count_numeric_cleaned,thread_count_unit,voltage_numeric_cleaned,weight_numeric_cleaned,weight_unit
0,True,"11 5, 2015",A8LUWTIPU9CZB,0560467893,1446681600,Home Dcor,Home Dcor Accents,Corner Shelves,WELLAND,"{'Features': 'floating shelf', 'Dimensions': '...",None,None,None,None,black,None,20 inch,floating shelf,None,None,None,None,None,None,corner shelf,None,None,None,None,None,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN,20.00,20.00,NaN,NaN,NaN,NaN,in,in,in,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,True,"05 7, 2015",A3B6GKQQ1JJ167,0560467893,1430956800,Home Dcor,Home Dcor Accents,Corner Shelves,WELLAND,"{'Features': 'floating shelf', 'Dimensions': '...",None,None,None,None,black,None,20 inch,floating shelf,None,None,None,None,None,None,corner shelf,None,None,None,None,None,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN,20.00,20.00,NaN,NaN,NaN,NaN,in,in,in,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,True,"01 22, 2014",A3MCTN65BU7XRA,0681795107,1390348800,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,Timolino,"{'Material': 'stainless', 'Product_Type': 'mug...",None,None,None,None,None,None,None,None,None,stainless,None,None,None,None,mug,None,None,None,None,None,double wall,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,True,"10 30, 2013",A7JVZFSXVY9RL,0681795107,1383091200,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,Timolino,"{'Material': 'stainless', 'Product_Type': 'mug...",None,None,None,None,None,None,None,None,None,stainless,None,None,None,None,mug,None,None,None,None,None,double wall,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,True,"09 20, 2013",A2RQ7VLAK1SHPU,0681795107,1379635200,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,Timolino,"{'Material': 'stainless', 'Product_Type': 'mug...",None,None,None,None,None,None,None,None,None,stainless,None,None,None,None,mug,None,None,None,None,None,double wall,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. User features

`data/df_user_features.pkl` — built by
`feature_extraction_workflow/extract_features_user.py`. **One row per purchase
event** with 22 features, each computed only from that user's purchases on
**strictly earlier days**, so no row can see its own or any later interaction.
See that module's section in `feature_extraction_workflow/README.md`.

### Why this is a positional concat, not a merge

The natural key is (user, item, date) — joining on `reviewerID` alone would be a
real leak, attaching every event's feature vector, including future ones, to
every row of that user.

But that triple **isn't unique in this log**: 501,704 rows share a
`(reviewerID, asin, unixReviewTime)` triple with at least one other row, so a
key merge is many-to-many and square-joins those groups into **+572,198 phantom
rows (+8.3%)**.

The pickle was generated from this exact CSV, in file order, dropping nothing —
so row *i* of the pickle already is row *i* of the log. Aligning by position
gives identical semantics with zero fanout.

The asserts below **verify** that alignment element-wise instead of assuming it.
They also confirm the row order survived the item join in §4. If the pickle is
ever regenerated from a different or reordered log, this fails loudly rather
than silently pairing the wrong user's history to a row.

**Expect ~23.5% of rows to have all-NaN user features.** That's a user's first
purchase (and anything on that same first day) — there is no prior history to
summarize. Intended, not a defect.

In [24]:
df_user_features = pd.read_pickle(DATA_DIR / "df_user_features.pkl")
print("df_user_features:", df_user_features.shape)
display(df_user_features.head(3))

# --- Verify positional alignment before relying on it ---
assert len(df_user_features) == len(df), (
    f"row count mismatch: user features {len(df_user_features):,} vs "
    f"interactions {len(df):,}"
)
for key in ("reviewerID", "asin", "unixReviewTime"):
    n_bad = int((df_user_features[key].to_numpy() != df[key].to_numpy()).sum())
    assert n_bad == 0, f"row order mismatch on {key}: {n_bad:,} rows differ"
print("alignment verified: reviewerID / asin / unixReviewTime match row-for-row\n")

# Attach the 22 feature columns (the key columns are already in df)
USER_FEATURE_COLS = [
    c for c in df_user_features.columns
    if c not in ("reviewerID", "asin", "reviewTime", "unixReviewTime")
]
df = pd.concat(
    [df.reset_index(drop=True),
     df_user_features[USER_FEATURE_COLS].reset_index(drop=True)],
    axis=1,
)

n_cold = int(df["prior_purchase_count"].isna().sum())
print(f"attached {len(USER_FEATURE_COLS)} user feature columns -> df {df.shape}")
print(f"cold-start rows (empty history, all user features NaN): "
      f"{n_cold:,} ({n_cold / len(df) * 100:.1f}%)")
df.head(5)

df_user_features: (6898955, 26)


,reviewerID,asin,reviewTime,unixReviewTime,prior_purchase_count,purchase_frequency,distinct_items,distinct_brands,distinct_categories,account_tenure,recency,days_since_first_purchase,avg_inter_purchase_gap,inter_purchase_gap_std,preferred_dow,preferred_month,preferred_season,activity_trend,favorite_cat_2,favorite_cat_3,favorite_cat_4,category_entropy,category_diversity,favorite_brand,brand_loyalty,brand_diversity
0,A8LUWTIPU9CZB,0560467893,"11 5, 2015",1446681600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,NaN
1,A3B6GKQQ1JJ167,0560467893,"05 7, 2015",1430956800,2.00,0.03,2.00,2.00,2.00,"1,598.00",452.00,"2,050.00","1,598.00",NaN,4.00,9.00,fall,0.00,Kitchen & Dining,Small Appliances,Juicers,1.00,1.00,Omega,0.50,1.00
2,A3MCTN65BU7XRA,0681795107,"01 22, 2014",1390348800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,NaN


alignment verified: reviewerID / asin / unixReviewTime match row-for-row

attached 22 user feature columns -> df (6898955, 81)
cold-start rows (empty history, all user features NaN): 1,624,415 (23.5%)


,verified,reviewTime,reviewerID,asin,unixReviewTime,cat_2,cat_3,cat_4_clean,brand,extracted_features,Bar_Pressure,Brand,Capacity_Cups,Capacity_Volume,Color,Density_Weight,Dimensions,Features,Filter_Rating,Material,Part_Number,Piece_Count,Pocket_Depth,Power_Rating,Product_Type,Scent,Shape,Shape_Style,Size,Stage_Count,Sub_Type,Theme,Thread_Count,Voltage,Weight,bar_pressure_numeric_cleaned,capacity_cups_numeric_cleaned,capacity_volume_numeric_cleaned,capacity_volume_unit,density_weight_lb_cleaned,dimension_1,dimension_1_in_cleaned,dimension_2,dimension_2_in_cleaned,dimension_3,dimension_3_in_cleaned,dimension_unit,dimension_unit_clean,dimension_unit_src,piece_count_numeric_cleaned,piece_count_unit,pocket_depth_in_cleaned,power_rating_w_cleaned,stage_count_numeric_cleaned,thread_count_numeric_cleaned,thread_count_unit,voltage_numeric_cleaned,weight_numeric_cleaned,weight_unit,prior_purchase_count,purchase_frequency,distinct_items,distinct_brands,distinct_categories,account_tenure,recency,days_since_first_purchase,avg_inter_purchase_gap,inter_purchase_gap_std,preferred_dow,preferred_month,preferred_season,activity_trend,favorite_cat_2,favorite_cat_3,favorite_cat_4,category_entropy,category_diversity,favorite_brand,brand_loyalty,brand_diversity
0,True,"11 5, 2015",A8LUWTIPU9CZB,0560467893,1446681600,Home Dcor,Home Dcor Accents,Corner Shelves,WELLAND,"{'Features': 'floating shelf', 'Dimensions': '...",None,None,None,None,black,None,20 inch,floating shelf,None,None,None,None,None,None,corner shelf,None,None,None,None,None,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN,20.00,20.00,NaN,NaN,NaN,NaN,in,in,in,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,NaN
1,True,"05 7, 2015",A3B6GKQQ1JJ167,0560467893,1430956800,Home Dcor,Home Dcor Accents,Corner Shelves,WELLAND,"{'Features': 'floating shelf', 'Dimensions': '...",None,None,None,None,black,None,20 inch,floating shelf,None,None,None,None,None,None,corner shelf,None,None,None,None,None,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN,20.00,20.00,NaN,NaN,NaN,NaN,in,in,in,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.00,0.03,2.00,2.00,2.00,"1,598.00",452.00,"2,050.00","1,598.00",NaN,4.00,9.00,fall,0.00,Kitchen & Dining,Small Appliances,Juicers,1.00,1.00,Omega,0.50,1.00
2,True,"01 22, 2014",A3MCTN65BU7XRA,0681795107,1390348800,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,Timolino,"{'Material': 'stainless', 'Product_Type': 'mug...",None,None,None,None,None,None,None,None,None,stainless,None,None,None,None,mug,None,None,None,None,None,double wall,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,NaN
3,True,"10 30, 2013",A7JVZFSXVY9RL,0681795107,1383091200,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,Timolino,"{'Material': 'stainless', 'Product_Type': 'mug...",None,None,None,None,None,None,None,None,None,stainless,None,None,None,None,mug,None,None,None,None,None,double wall,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.00,0.03,3.00,2.00,2.00,"2,158.00",739.00,"2,897.00","1,079.00",695.00,3.00,10.00,fall,0.00,Kitchen & Dining,Small Appliances,Blenders,1.00,0.67,Hamilton Beach,0.50,0.67
4,True,"09 20, 2013",A2RQ7VLAK1SHPU,0681795107,1379635200,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,Timolino,"{'Material': 'stainless', 'Product_Type': 'mug...",None,None,None,None,None,None,None,None,None,stainless,None,None,None,None,mug,None,None,None,None,None,double wall,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,NaN


## 5. Every column in the final dataset

`df` is now the full table: one row per interaction, carrying the interaction
keys, the item features joined on `asin`, and the user features aligned by
position.

The inventory below is the list to work from when deciding what the towers
actually consume — `source`, `dtype`, coverage, distinct count and a few example
values per column. `source` is usually the first thing that settles a column's
fate: interaction columns are known at serving time, item columns describe the
catalogue, user columns are the leakage-safe history.

In [25]:
# --- Every column in the final dataset, with what you need to judge it ------
# `source` says where a column came from, which is usually the first thing that
# decides its fate: an interaction column is known at serving time, an item
# column describes the catalogue, a user column is the leakage-safe history.

INTERACTION_COLS = set(df_reviews.columns)
USER_COLS = set(USER_FEATURE_COLS)


def _source(col):
    if col in USER_COLS:
        return "user"
    if col in INTERACTION_COLS:
        return "interaction"
    return "item"


def column_inventory(frame=None, sample=3):
    frame = df if frame is None else frame
    rows = []
    for c in frame.columns:
        s = frame[c]
        nn = int(s.notna().sum())
        try:
            distinct = int(s.nunique(dropna=True))
        except TypeError:                      # dict/list columns
            distinct = np.nan
        try:
            vals = s.dropna().unique()[:sample]
            example = ", ".join(str(v)[:28] for v in vals)
        except Exception:
            example = ""
        rows.append({
            "column": c,
            "source": _source(c),
            "dtype": str(s.dtype),
            "non_null": nn,
            "coverage_%": nn / len(frame) * 100,
            "distinct": distinct,
            "example": example[:64],
        })
    out = (pd.DataFrame(rows)
             .sort_values(["source", "coverage_%"], ascending=[True, False])
             .reset_index(drop=True))
    return out


inventory = column_inventory()
print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")
print(inventory.groupby("source")["column"].count().to_string(), "\n")
display(inventory)

6,898,955 rows x 81 columns
source
interaction     5
item           54
user           22 



,column,source,dtype,non_null,coverage_%,distinct,example
0,verified,interaction,bool,6898955,100.00,2.00,"True, False"
1,reviewTime,interaction,object,6898955,100.00,"6,349.00","11 5, 2015, 05 7, 2015, 01 22, 2014"
2,reviewerID,interaction,object,6898955,100.00,"777,242.00","A8LUWTIPU9CZB, A3B6GKQQ1JJ167, A3MCTN65BU7XRA"
3,asin,interaction,object,6898955,100.00,"189,172.00","0560467893, 0681795107, 0768205921"
4,unixReviewTime,interaction,int64,6898955,100.00,"6,349.00","1446681600, 1430956800, 1390348800"
...,...,...,...,...,...,...,...
76,brand_loyalty,user,float64,5051470,73.22,"3,423.00","0.5, 1.0, 0.07142857142857142"
77,favorite_cat_4,user,object,4968569,72.02,447.00,"Juicers, Blenders, Waffle Irons"
78,avg_inter_purchase_gap,user,float64,4439568,64.35,"131,669.00","1598.0, 1079.0, 49.53333333333333"
79,activity_trend,user,float64,3908755,56.66,"1,846,648.00","0.0006257822277847309, 0.0008141864122172733, ..."


## 6. Train / test split

Same convention as `functions/interaction_matrix.py` and `adding_bpr.ipynb`: one
global time cutoff, train strictly before it, test at or after, and items kept
only where at least `MIN_USERS` distinct users interacted with them.

The cutoff itself lives in `ttn/constants.json` as `date_threshold`, not in
this cell. `complementary_cats_pairs/pairs.ipynb` reads the same file for the
same purpose, so the split the model trains on and the split the co-purchase
pairs are built from are the same value by construction rather than by
convention — edit the JSON and both notebooks follow.

Two properties worth keeping:

- **A global cutoff, not a per-user one.** Every training row precedes every
  test row, so nothing in train can depend on a future the model would not have
  had.
- **The item filter is fitted on train only.** Choosing which items exist using
  test rows would leak the evaluation period into the vocabulary — the same rule
  the user-feature pipeline follows.

Transformations you want on the features (encoding, bucketing, scaling) belong
between §5 and here, so they can be fitted on `train` alone.

In [ ]:
# --- Train / test split ----------------------------------------------------
# Same convention as `functions/interaction_matrix.py` and adding_bpr.ipynb:
# a single global time cutoff, train strictly before it, test at or after it,
# and items kept only if at least `MIN_USERS` distinct users interacted with
# them. A global cutoff rather than a per-user one keeps the split honest —
# every training row precedes every test row, so nothing in train can depend on
# a future the model would not have had.

MIN_USERS = 5           # matches InteractionMatrixBuilder's default

# The split point is not derived here — it is read from `ttn/constants.json`,
# the one place it is written down. `complementary_cats_pairs/pairs.ipynb`
# reads the same file, so the two cannot drift apart.
constants = json.loads(Path("constants.json").read_text())
DATE_THRESHOLD = constants["date_threshold"]

# Naive timestamps are treated as UTC, which is the basis unixReviewTime is on.
cutoff_time = pd.Timestamp(DATE_THRESHOLD).timestamp()
print(f"cutoff: {DATE_THRESHOLD} (unix {int(cutoff_time):,}), "
      f"from ttn/constants.json")

# Item filter is fitted on TRAIN ONLY — deciding which items exist using test
# rows would leak the evaluation period into the vocabulary.
train = df[df["unixReviewTime"] < cutoff_time].copy()
test = df[df["unixReviewTime"] >= cutoff_time].copy()

item_users = train.groupby("asin")["reviewerID"].nunique()
keep_items = item_users[item_users >= MIN_USERS].index
train = train[train["asin"].isin(keep_items)]

# A test row is only scoreable if both its user and its item were seen in train.
test = test[test["asin"].isin(keep_items) & test["reviewerID"].isin(train["reviewerID"].unique())]

print(f"\ntrain : {len(train):>10,} rows | {train['reviewerID'].nunique():>8,} users "
      f"| {train['asin'].nunique():>7,} items")
print(f"test  : {len(test):>10,} rows | {test['reviewerID'].nunique():>8,} users "
      f"| {test['asin'].nunique():>7,} items")
print(f"\nitems dropped by the >= {MIN_USERS}-user filter: "
      f"{df['asin'].nunique() - len(keep_items):,} of {df['asin'].nunique():,}")
print(f"test rows dropped as unscoreable (cold user or item): "
      f"{len(df[df['unixReviewTime'] >= cutoff_time]) - len(test):,}")

In [27]:
train.head(3)

,verified,reviewTime,reviewerID,asin,unixReviewTime,cat_2,cat_3,cat_4_clean,brand,extracted_features,Bar_Pressure,Brand,Capacity_Cups,Capacity_Volume,Color,Density_Weight,Dimensions,Features,Filter_Rating,Material,Part_Number,Piece_Count,Pocket_Depth,Power_Rating,Product_Type,Scent,Shape,Shape_Style,Size,Stage_Count,Sub_Type,Theme,Thread_Count,Voltage,Weight,bar_pressure_numeric_cleaned,capacity_cups_numeric_cleaned,capacity_volume_numeric_cleaned,capacity_volume_unit,density_weight_lb_cleaned,dimension_1,dimension_1_in_cleaned,dimension_2,dimension_2_in_cleaned,dimension_3,dimension_3_in_cleaned,dimension_unit,dimension_unit_clean,dimension_unit_src,piece_count_numeric_cleaned,piece_count_unit,pocket_depth_in_cleaned,power_rating_w_cleaned,stage_count_numeric_cleaned,thread_count_numeric_cleaned,thread_count_unit,voltage_numeric_cleaned,weight_numeric_cleaned,weight_unit,prior_purchase_count,purchase_frequency,distinct_items,distinct_brands,distinct_categories,account_tenure,recency,days_since_first_purchase,avg_inter_purchase_gap,inter_purchase_gap_std,preferred_dow,preferred_month,preferred_season,activity_trend,favorite_cat_2,favorite_cat_3,favorite_cat_4,category_entropy,category_diversity,favorite_brand,brand_loyalty,brand_diversity
0,True,"11 5, 2015",A8LUWTIPU9CZB,0560467893,1446681600,Home Dcor,Home Dcor Accents,Corner Shelves,WELLAND,"{'Features': 'floating shelf', 'Dimensions': '...",None,None,None,None,black,None,20 inch,floating shelf,None,None,None,None,None,None,corner shelf,None,None,None,None,None,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN,20.00,20.00,NaN,NaN,NaN,NaN,in,in,in,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,NaN
1,True,"05 7, 2015",A3B6GKQQ1JJ167,0560467893,1430956800,Home Dcor,Home Dcor Accents,Corner Shelves,WELLAND,"{'Features': 'floating shelf', 'Dimensions': '...",None,None,None,None,black,None,20 inch,floating shelf,None,None,None,None,None,None,corner shelf,None,None,None,None,None,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN,20.00,20.00,NaN,NaN,NaN,NaN,in,in,in,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.00,0.03,2.00,2.00,2.00,"1,598.00",452.00,"2,050.00","1,598.00",NaN,4.00,9.00,fall,0.00,Kitchen & Dining,Small Appliances,Juicers,1.00,1.00,Omega,0.50,1.00
2,True,"01 22, 2014",A3MCTN65BU7XRA,0681795107,1390348800,Kitchen & Dining,Travel & To-Go Drinkware,Commuter & Travel Mugs,Timolino,"{'Material': 'stainless', 'Product_Type': 'mug...",None,None,None,None,None,None,None,None,None,stainless,None,None,None,None,mug,None,None,None,None,None,double wall,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,None,None,None,NaN,NaN,None,NaN,NaN


## 7. Complementary category pairs

`data/complementary_categories.pkl` — built by `complementary_cats_pairs/`, which
turns Amazon's `also_buy` lists into **directed** category pairs scored by
support and lift, then keeps the ones above the chosen thresholds
(`MIN_EDGES = 5`, `MIN_LIFT = 2.0`). It replaces the hand-curated
`data/complementary_category_map.csv`, which is no longer produced.

One row per surviving `(source path -> target path)` pair:

| Column | Meaning |
| --- | --- |
| `src_cat_2`, `src_cat_3`, `src_cat_4` | source category path (`cat_4` folded through `category_taxonomy.json`) |
| `dst_cat_2`, `dst_cat_3`, `dst_cat_4` | target category path, folded the same way |
| `edges` | co-purchase edges behind this pair |
| `src_edges`, `dst_edges` | the two marginals the lift is measured against |
| `support` | `edges / N`, the share of resolved co-purchase traffic |
| `lift` | `P(s, d) / (P(s) P(d))`; `1.0` is independence |

`A -> B` and `B -> A` are separate rows — `also_buy` is listed per source item,
and the two directions can carry very different traffic. The six category
columns are `category` dtype, which matters when comparing two of them below.

This is built off the **whole** log, so it is an analysis artifact rather than
a training feature as it stands.

The item-level counterpart — every pair of items the same user actually bought,
rather than the categories Amazon links — is built by
`complementary_cats_pairs/pairs.ipynb` and saved to
`data/co_purchase_pairs.pkl`. That one *is* cutoff-filtered, so it is safe to
train on; read it with `pd.read_pickle` if you need it here.

In [28]:
# Complementary category pairs, one row per directed (source -> target) pair.
# DATA_DIR is the repo-root `data/`, set in section 1.
comp_cat = pd.read_pickle(DATA_DIR / "complementary_categories.pkl")

n_src = comp_cat.groupby(["src_cat_2", "src_cat_3", "src_cat_4"],
                         observed=True).ngroups
print("comp_cat:", comp_cat.shape)
print("columns:", list(comp_cat.columns))
print(f"pairs: {len(comp_cat):,} | distinct source paths: {n_src:,} | "
      f"edges behind them: {comp_cat['edges'].sum():,}")
comp_cat.head(5)

comp_cat: (5845, 11)
columns: ['src_cat_2', 'src_cat_3', 'src_cat_4', 'dst_cat_2', 'dst_cat_3', 'dst_cat_4', 'edges', 'src_edges', 'dst_edges', 'support', 'lift']
pairs: 5,845 | distinct source paths: 455 | edges behind them: 807,578


,src_cat_2,src_cat_3,src_cat_4,dst_cat_2,dst_cat_3,dst_cat_4,edges,src_edges,dst_edges,support,lift
0,Home Dcor,Home Dcor Accents,Ornaments,Home Dcor,Home Dcor Accents,Ornaments,40095,48426,44936,0.04,18.92
1,Wall Art,Posters & Prints,Missing,Wall Art,Posters & Prints,Missing,38813,47710,45745,0.04,18.26
2,Home Dcor,Home Dcor Accents,Collectible Figurines,Home Dcor,Home Dcor Accents,Collectible Figurines,34984,45771,46396,0.03,16.92
3,Kitchen & Dining,Bakeware,Baking Tools & Accessories,Kitchen & Dining,Bakeware,Baking Tools & Accessories,31348,47962,43909,0.03,15.29
4,Home Dcor,Home Dcor Accents,Decorative Accessories,Home Dcor,Home Dcor Accents,Decorative Accessories,27188,40206,41919,0.03,16.57


In [29]:
# Everything Living Room Furniture is bought alongside, strongest pair first.
comp_cat[(comp_cat["src_cat_2"] == "Furniture") &
         (comp_cat["src_cat_3"] == "Living Room Furniture")]

,src_cat_2,src_cat_3,src_cat_4,dst_cat_2,dst_cat_3,dst_cat_4,edges,src_edges,dst_edges,support,lift
57,Furniture,Living Room Furniture,Tables,Furniture,Living Room Furniture,Tables,1697,4115,3421,0.00,123.79
228,Furniture,Living Room Furniture,TV & Media Furniture,Furniture,Living Room Furniture,Tables,348,1364,3421,0.00,76.58
236,Furniture,Living Room Furniture,Tables,Furniture,Living Room Furniture,TV & Media Furniture,337,4115,1148,0.00,73.25
297,Furniture,Living Room Furniture,TV & Media Furniture,Furniture,Living Room Furniture,TV & Media Furniture,273,1364,1148,0.00,179.03
523,Furniture,Living Room Furniture,TV & Media Furniture,Furniture,Home Office Furniture,Bookcases,161,1364,1329,0.00,91.20
...,...,...,...,...,...,...,...,...,...,...,...
5711,Furniture,Living Room Furniture,Chairs,Home Dcor,"Area Rugs, Runners & Pads",Area Rugs,5,451,3004,0.00,3.79
5763,Furniture,Living Room Furniture,Chairs,Furniture,Home Office Furniture,Bookcases,5,451,1329,0.00,8.57
5779,Furniture,Living Room Furniture,Chairs,Furniture,Kids' Furniture,"Bookcases, Cabinets & Shelves",5,451,288,0.00,39.53
5792,Furniture,Living Room Furniture,Tables,Furniture,Accent Furniture,Display & Curio Cabinets,5,4115,24,0.00,51.99


In [30]:
# Pairs that stay inside their own cat_3 — a chair listed with another chair.
# These are kept by design; filter them out here if you need strict complements.
#
# `.astype(str)` is required, not cosmetic: both columns are `category` dtype
# with *different* category sets (69 source vs 141 target values), and pandas
# refuses to compare two Categoricals unless their categories match.
comp_cat[comp_cat["src_cat_3"].astype(str) == comp_cat["dst_cat_3"].astype(str)]

,src_cat_2,src_cat_3,src_cat_4,dst_cat_2,dst_cat_3,dst_cat_4,edges,src_edges,dst_edges,support,lift
0,Home Dcor,Home Dcor Accents,Ornaments,Home Dcor,Home Dcor Accents,Ornaments,40095,48426,44936,0.04,18.92
1,Wall Art,Posters & Prints,Missing,Wall Art,Posters & Prints,Missing,38813,47710,45745,0.04,18.26
2,Home Dcor,Home Dcor Accents,Collectible Figurines,Home Dcor,Home Dcor Accents,Collectible Figurines,34984,45771,46396,0.03,16.92
3,Kitchen & Dining,Bakeware,Baking Tools & Accessories,Kitchen & Dining,Bakeware,Baking Tools & Accessories,31348,47962,43909,0.03,15.29
4,Home Dcor,Home Dcor Accents,Decorative Accessories,Home Dcor,Home Dcor Accents,Decorative Accessories,27188,40206,41919,0.03,16.57
...,...,...,...,...,...,...,...,...,...,...,...
5833,Furniture,Accent Furniture,Room Dividers,Furniture,Accent Furniture,Missing,5,106,16,0.00,"3,027.34"
5834,Kitchen & Dining,Small Appliance Parts & Accessories,Food Processor Parts & Accessories,Kitchen & Dining,Small Appliance Parts & Accessories,Coffee & Espresso Machine Parts & Accessories,5,865,1885,0.00,3.15
5836,Home Dcor,Picture Frames,Clip Photo Holders,Home Dcor,Picture Frames,Wall & Tabletop Frames,5,26,8130,0.00,24.29
5841,Kitchen & Dining,Small Appliances,Blenders,Kitchen & Dining,Small Appliances,Rice Cookers,5,1375,374,0.00,9.98


## 8. Training pairs: co-purchases that follow a complementary mapping

The positives the model trains on. Two ingredients, joined here:

- `data/co_purchase_pairs.pkl` (§ built by `complementary_cats_pairs/pairs.ipynb`)
  — every pair of items one user bought within 90 days of each other, before
  `date_threshold`. This is *what people actually bought together*, and it
  includes plenty of pairs that are not complements: two near-identical mugs,
  a duplicate purchase, coincidence.
- `data/complementary_categories.pkl` (§7) — which **category** buys into which,
  scored by support and lift.

Joining item categories onto both ends and keeping only the pairs whose
categories are a known complementary relation is the paper's "complementarity
relation heuristic": it filters co-purchase down to co-purchase *that looks
complementary at the category level*.

**Direction is deliberately ignored at this stage.** `co_purchase_pairs` stores
each pair with the alphabetically smaller asin in `asinA`, which carries no
meaning, while the mapping is directed. So a pair is kept when **either**
`A → B` **or** `B → A` appears in the mapping. Turning these into ordered
(query, candidate) examples is the next step.

### Why `cat_4_clean` and not `cat_4`

The mapping's `src_cat_4` / `dst_cat_4` are folded through
`category_taxonomy.json` — the same folding §2 applies to build `cat_4_clean`
(verified identical to the package's `fold_cat_4`, 451 distinct values either
way). Joining on the raw `cat_4` instead would fail on naming, not on absence:
only 83.9% of items land on a known source path rather than 99.8%, and about
240k pairs are lost for no real reason.


In [ ]:
# --- Co-purchase pairs, with the category path of each end ------------------
CAT_LEVELS = ["cat_2", "cat_3", "cat_4_clean"]   # matches the mapping's levels

co_pairs = pd.read_pickle(DATA_DIR / "co_purchase_pairs.pkl")
print(f"co_purchase pairs        : {len(co_pairs):,}")

# One integer id per distinct (cat_2, cat_3, cat_4) path, in a vocabulary
# shared by the items and by both ends of the mapping — so a path comparison
# is an integer comparison rather than three string comparisons.
SEP = "\x1f"
item_path = df_features[CAT_LEVELS].astype(str).agg(SEP.join, axis=1)
src_path = comp_cat[SRC_COLS].astype(str).agg(SEP.join, axis=1)
dst_path = comp_cat[DST_COLS].astype(str).agg(SEP.join, axis=1)

vocabulary = pd.Index(pd.unique(
    pd.concat([item_path, src_path, dst_path], ignore_index=True)))
n_paths = len(vocabulary)
asin_to_path = pd.Series(vocabulary.get_indexer(item_path),
                         index=df_features["asin"].to_numpy())

a_path = asin_to_path.reindex(co_pairs["asinA"].astype(str)).to_numpy()
b_path = asin_to_path.reindex(co_pairs["asinB"].astype(str)).to_numpy()

# An asin with no row in df_features has no category path, so it cannot be
# tested against the mapping at all.
joinable = ~(np.isnan(a_path) | np.isnan(b_path))
a_id = np.where(joinable, np.nan_to_num(a_path, nan=-1), -1).astype(np.int64)
b_id = np.where(joinable, np.nan_to_num(b_path, nan=-1), -1).astype(np.int64)

# Every directed (source path -> target path) the mapping allows, as one int.
allowed = np.unique(vocabulary.get_indexer(src_path).astype(np.int64) * n_paths
                    + vocabulary.get_indexer(dst_path).astype(np.int64))

a_to_b = np.isin(a_id * n_paths + b_id, allowed) & joinable
b_to_a = np.isin(b_id * n_paths + a_id, allowed) & joinable
keep = a_to_b | b_to_a

print(f"  both asins in features : {joinable.sum():>12,} ({joinable.mean():.1%})")
print(f"  dropped, no features   : {(~joinable).sum():>12,}")
print(f"\nkept (either direction)  : {keep.sum():>12,} "
      f"({keep.sum() / len(co_pairs):.1%} of all, "
      f"{keep.sum() / joinable.sum():.1%} of joinable)")
print(f"  A -> B only            : {(a_to_b & ~b_to_a).sum():>12,}")
print(f"  B -> A only            : {(b_to_a & ~a_to_b).sum():>12,}")
print(f"  both directions        : {(a_to_b & b_to_a).sum():>12,}")

### The table

Eight columns: the two asins, and the three category levels of each. `a_*`
describes the item in `asinA`, `b_*` the item in `asinB` — still unordered, so
neither is yet the query.

`a_to_b` / `b_to_a` are carried through because the next step needs them: they
are exactly which orientation of the pair the mapping licenses, so turning this
into directed (query, candidate) rows is a matter of selecting on them rather
than re-deriving anything.


In [ ]:
# The vocabulary is a few hundred paths, so split it once and index into it
# rather than splitting 1.6M strings twice.
path_levels = pd.DataFrame([p.split(SEP) for p in vocabulary], columns=CAT_LEVELS)

a_levels = path_levels.iloc[a_id[keep]].reset_index(drop=True).add_prefix("a_")
b_levels = path_levels.iloc[b_id[keep]].reset_index(drop=True).add_prefix("b_")

training_pairs = pd.concat(
    [co_pairs.loc[keep, ["asinA", "asinB"]].reset_index(drop=True),
     a_levels, b_levels,
     pd.DataFrame({"a_to_b": a_to_b[keep], "b_to_a": b_to_a[keep]})],
    axis=1,
)
for c in a_levels.columns.tolist() + b_levels.columns.tolist():
    training_pairs[c] = training_pairs[c].astype("category")

print(f"training_pairs : {training_pairs.shape}")
print(f"columns        : {list(training_pairs.columns)}")
print(f"distinct items : "
      f"{pd.concat([training_pairs['asinA'], training_pairs['asinB']]).nunique():,}")
print(f"memory         : {training_pairs.memory_usage(deep=True).sum() / 1e6:,.0f} MB")
training_pairs.head(10)